In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install ultralytics

# **LOADING DATASET FROM DRIVE**

In [ ]:
import os
import random
import shutil

source_dir = "/content/drive/MyDrive/thermal_data/Thermal_Camera"
train_dir = "/content/drive/MyDrive/thermal_data/Thermal_Camera/train"
val_dir = "/content/drive/MyDrive/thermal_data/Thermal_Camera/val"

classes = ["Smoke", "Perfume", "NoGas", "Mixture"]

for cls in classes:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)

    images = os.listdir(os.path.join(source_dir, cls))
    random.shuffle(images)

    split = int(0.8 * len(images))
    train_imgs = images[:split]
    val_imgs = images[split:]

    for img in train_imgs:
        shutil.copy(
            os.path.join(source_dir, cls, img),
            os.path.join(train_dir, cls, img)
        )

    for img in val_imgs:
        shutil.copy(
            os.path.join(source_dir, cls, img),
            os.path.join(val_dir, cls, img)
        )

# **TRAINING OF YOLOv8 MODEL**

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")  # classification model

model.train(
    data= "/content/drive/MyDrive/thermal_data/Thermal_Camera",
    epochs=50,
    imgsz=224,
    batch=32,
    device=0
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/thermal_data/Thermal_Camera, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d4c8d19b140>
curves: []
curves_results: []
fitness: 0.994140625
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.98828125, 'metrics/accuracy_top5': 1.0, 'fitness': 0.994140625}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.06803151796503926, 'inference': 0.23713413905994685, 'loss': 0.004280989844573924, 'postprocess': 0.0003500835958902826}
task: 'classify'
top1: 0.98828125
top5: 1.0

# **EXPORTING AND SAVING OF YOLOv8 MODEL**

In [ ]:
import shutil

source = "/content/runs/classify/train/weights/best.pt"
destination = "/content/drive/MyDrive/thermal_data/yolov8_gas_classifier.pt"

shutil.copy(source, destination)

print("Model saved to Google Drive successfully.")

Model saved to Google Drive successfully.


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/thermal_data/yolov8_gas_classifier.pt")

model.export(format="torchscript")

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n-cls summary (fused): 30 layers, 1,440,004 parameters, 0 gradients, 3.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/thermal_data/yolov8_gas_classifier.pt' with input shape (1, 3, 224, 224) BCHW and output shape(s) (1, 4) (2.8 MB)

TorchScript: starting export with torch 2.9.0+cu128...
TorchScript: export success ✅ 0.8s, saved as '/content/drive/MyDrive/thermal_data/yolov8_gas_classifier.torchscript' (5.6 MB)

Export complete (1.0s)
Results saved to /content/drive/MyDrive/thermal_data
Predict:         yolo predict task=classify model=/content/drive/MyDrive/thermal_data/yolov8_gas_classifier.torchscript imgsz=224 
Validate:        yolo val task=classify model=/content/drive/MyDrive/thermal_data/yolov8_gas_classifier.torchscript imgsz=224 data=/con

'/content/drive/MyDrive/thermal_data/yolov8_gas_classifier.torchscript'